# 1.LLaMA-Factory微调数据集制作

[LLaMA Factory网址](https://llamafactory.readthedocs.io/zh-cn/latest/index.html)

指令监督微调数据集
样例数据集： 

[指令监督微调样例数据集](https://github.com/hiyouga/LLaMA-Factory/blob/main/data/alpaca_zh_demo.json/)

指令监督微调(Instruct Tuning)通过让模型学习详细的指令以及对应的回答来优化模型在特定指令下的表现。

instruction 列对应的内容为人类指令， input 列对应的内容为人类输入， output 列对应的内容为模型回答。下面是一个例子


## 单轮对话

instruction-代表提问的问题
input-补充内容，如果没有要补充的内容可以为空
output-模型的回复

"alpaca_zh_demo.json"
{
  "instruction": "计算这些物品的总费用。 ",
  "input": "输入：汽车 - $3000，衣服 - $100，书 - $20。",
  "output": "汽车、衣服和书的总费用为 $3000 + $100 + $20 = $3120。"
},

## 多轮对话
下面提供一个 alpaca 格式 多轮 对话的例子，对于单轮对话只需省略 history 列即可。

[
  {
    "instruction": "人类指令（必填）",
    "input": "人类输入（选填）",
    "output": "模型回答（必填）",
    "system": "系统提示词（选填）",
    "history": [
      ["第一轮指令（选填）", "第一轮回答（选填）"],
      ["第二轮指令（选填）", "第二轮回答（选填）"]
    ]
  }
]

![](Image/2025-03-27-22-43-30.png)

## 弱智吧问答数据

![](Image/2025-03-27-22-49-07.png)

数据预览

![](Image/2025-03-27-22-51-52.png)

数据下载

![](Image/2025-03-27-22-53-28.png)

![](Image/2025-03-27-22-55-44.png)

## 数据转换


数据集转换可以上DeepSeek帮忙写Python脚本，然后自己进行微调

![](Image/2025-03-27-23-14-30.png)

In [1]:
import json

# 读取原始JSON文件
input_file = "data/ruozhiba_qaswift.json"  # 你的JSON文件名
output_file = "data/ruozhiba_qaswift_train.json"  # 输出的JSON文件名

with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)

# 转换后的数据
converted_data = []

for item in data:
    converted_item = {
        "instruction": item["query"],
        "input": "",
        "output": item["response"]
    }
    converted_data.append(converted_item)

# 保存为JSON文件（最外层是列表）
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=4)

print(f"转换完成，数据已保存为 {output_file}")

转换完成，数据已保存为 data/ruozhiba_qaswift_train.json


![](Image/2025-03-27-23-17-56.png)

## LLaMA-Factory数据集设置

将数据集上传到LLaMA-Factory的data目录下

![](Image/2025-03-27-23-31-04.png)

在dataset_info.json中配置数据集

![](Image/2025-03-27-23-36-37.png)

![](Image/2025-03-27-23-39-11.png)

# 2.使用 open-webui部署模型

首先进入LLaMa-Factory根目录，然后启动，否则在界面中会找不到数据集

llamafactory-cli webui

![](Image/2025-03-27-23-50-34.png)

此时会打开浏览器，对以下参数进行设置


![](Image/2025-03-28-23-21-37.png)

在生成模型里面，验证集可写可不写，因为生成模型几乎不可能过拟合，验证集就是测试集，测试验证有单独的栏目“Evaluate & Predict”来开展。

![](Image/2025-03-28-23-23-21.png)

开始训练

![](Image/2025-03-28-23-29-24.png)

![](Image/2025-03-28-23-31-32.png)

nvitop查看显存占用率，根据显存占用率调整批次大小

![](Image/2025-03-28-23-34-39.png)

![](Image/2025-03-28-23-32-53.png)

# 3.Lora模型合并与量化导出

## 训练评估

### 模型训练好的标志

模型的loss收敛

![](Image/2025-03-29-00-17-08.png)

### 权重保存路径

![](Image/2025-03-28-23-51-09.png)

### Evaluate & Predict-页面参数设置

![](Image/2025-03-29-00-01-58.png)

先中断训练，再开始评估

### 根据提示安装缺失的包

分词工具

![](Image/2025-03-29-00-10-37.png)

pip install jieba

![](Image/2025-03-29-00-13-30.png)

![](Image/2025-03-29-00-18-31.png)

pip install nltk

![](Image/2025-03-29-00-19-54.png)

In [ ]:
pip install rouge_chinese

### 验证集生成的评估结果

生成模型的这个评估结果只能做参考，主要验证方式还是靠问问题。

![](Image/2025-03-29-00-27-36.png)

### chat界面验证

![](Image/2025-03-29-00-33-13.png)

能够识别到数据集中的内容

![](Image/2025-03-29-00-34-13.png)

### 单轮对话注意清空历史

本次训练针对的是单轮对话进行训练，没有清空历史会导致回答和训练集里面的数据不匹配，没有清空历史回答效果如下：

![](Image/2025-03-29-00-39-19.png)

上面的回答和数据集里面的毫不相干

![](Image/2025-03-29-00-40-15.png)

![](Image/2025-03-29-00-41-55.png)

### 模型导出

![](Image/2025-03-29-00-48-48.png)

![](Image/2025-03-29-00-50-46.png)

### 验证导出的模型


![](Image/2025-03-29-00-53-41.png)

打包后的模型回答和训练好的数据集结果一致

![](Image/2025-03-29-00-55-52.png)